# 6.5 - Multimodal Models: Give Your LLM Eyes

**Module 6 - Image & Multimodal Generation** | Netsetos GenAI Engineering

This notebook is the working code for a vision-language model (VLM): an LLM you send an image plus text to, and get text back. Every cell is that one shape - image + text in, text out - applied to image Q&A, structured document extraction, multi-image comparison, and a self-hosted open model.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough. (The API cells call Google's `gemini-3` models and the last one loads an open model on a GPU, so they need a key/hardware to actually run - the walkthroughs tell you what each would print.)

## Setup

A single marker comment that pins down the notebook's reproducibility contract - the lesson's examples lean on seeded random values so that a given cell prints the same thing every run. There is nothing to install or import here; the real environment prep (the SDK client) is the next cell.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a bookkeeping cell, not a computation. It documents that any randomness in the lesson is seeded so outputs are stable across runs, and it serves as the top-of-notebook anchor before the client is built.

**How the code works - step by step**
- **`# Reproducibility ...`** - a lone comment; no code executes. It flags that seeded values are used throughout so results are deterministic.

**In one line:** a no-op marker that promises repeatable outputs.

**What you'll see:** (no output - environment prep)

## 1 - Set up the one client you reuse all lesson

Every later cell talks to a VLM, and they all reuse the `client` built here. The key idea to carry forward: with the modern unified **google-genai** SDK, an image is just another "part" you drop into the request next to your text - so image Q&A looks almost exactly like a normal text call.

> **Needs a Google API key** (set `GOOGLE_API_KEY`) and a `receipt.jpg` file. No GPU or model download - this hits a hosted model.

In [ ]:
# pip install google-genai pillow transformers
from google import genai
from PIL import Image
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# An image is just another "part" in the request, next to your text.
img = Image.open("receipt.jpg")
resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=["What is the total and the date on this receipt?", img],
)
print(resp.text)
# Output: The total is Rs. 1,240 and the date is 12 March 2026.

**Explanation**

This is the whole API surface of the lesson in one call. It creates a reusable hosted-model client, opens an image with PIL, and passes a list mixing a question and that image to `generate_content` - the image + text in, text out shape. Read it as the template every other cell specializes.

**How the code works - step by step**
- **`# pip install google-genai pillow transformers`** - the packages the notebook needs; `google-genai` is the current SDK (the old `google.generativeai` was deprecated in 2025).
- **`from google import genai` / `from PIL import Image`** - the SDK and the image loader.
- **`client = genai.Client(api_key=...)`** - one reusable client, reading the key from the environment. The deprecated package used a global `configure()` instead.
- **`img = Image.open("receipt.jpg")`** - loads the image as a PIL object; the SDK handles uploading and encoding it.
- **`client.models.generate_content(model="gemini-3-flash", contents=[question, img])`** - the core call: `contents` is a list mixing text and the image. `gemini-3-flash` is the fast, cheap tier.
- **`print(resp.text)`** - the model's plain-text answer.

**In one line:** build one client, then send `[text, image]` and read `resp.text`.

**What you'll see:** The model reads the receipt and prints a one-line answer: "The total is Rs. 1,240 and the date is 12 March 2026."

## 2 - Image Q&A: the prompt is the control surface

Same call as the setup - `contents=[prompt, image]` - but now the prompt does the real work. A role, a specific task, and a required output format turn a vague "describe this" into an actionable list, exactly like prompting a text LLM.

> **Needs a Google API key** and a `shelf.jpg` file.

In [ ]:
# reuse `client` and PIL Image from the setup
shelf = Image.open("shelf.jpg")

prompt = (
    "You are a retail auditor. Look at this shelf photo and list every product "
    "that is out of stock or clearly misplaced. One item per line; if none, say 'all good'."
)
resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=[prompt, shelf],
)
print(resp.text)
# Output: - Shelf 2: cereal slot empty (out of stock)
# Output: - Shelf 3: shampoo bottle in the snacks row (misplaced)

**Explanation**

This cell shows that image Q&A is not a new API - it is the setup call with a better-engineered prompt. It reuses the existing `client`, loads a shelf photo, and asks a role-framed, format-constrained question so the answer comes back as a clean per-item list instead of prose.

**How the code works - step by step**
- **`shelf = Image.open("shelf.jpg")`** - loads the image; `client` is reused from Cell 1.
- **`prompt = (...)`** - the control surface: a role ("retail auditor"), a precise task (list out-of-stock or misplaced products), and an output format (one item per line, or 'all good').
- **`generate_content(model="gemini-3-flash", contents=[prompt, shelf])`** - identical shape to the setup; only the prompt changed.
- **`print(resp.text)`** - the structured-by-instruction answer.

**In one line:** same call as Cell 1, but role + task + output-format in the prompt shapes the answer.

**What you'll see:** A short line-per-item audit list, e.g. "- Shelf 2: cereal slot empty (out of stock)" and "- Shelf 3: shampoo bottle in the snacks row (misplaced)".

## 3 - Document extraction into typed JSON

Instead of asking for prose about a receipt, hand the model a schema and make it fill the boxes. This is Lesson 5.5's structured-output trick applied over an image, plus one critical extra field - `legible` - so the model can flag what it could not read instead of inventing a number.

> **Needs a Google API key** and a `receipt.jpg` file.

In [ ]:
from google.genai import types
from pydantic import BaseModel
from PIL import Image
# reuse `client` from the setup

class Receipt(BaseModel):
    vendor: str
    date: str            # ISO 8601, e.g. "2026-03-12"
    total: float
    legible: bool       # False if the model could not read it clearly

resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=["Extract the receipt fields. Set legible=false if unsure.", Image.open("receipt.jpg")],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Receipt,          # force the reply to match the schema
    ),
)
data = Receipt.model_validate_json(resp.text)
print(data.vendor, data.total, data.legible)
# Output: Sunrise Grocers 1240.0 True

**Explanation**

This cell turns image Q&A into reliable document AI by forcing the reply to match a Pydantic model. It defines a `Receipt` schema, passes it as `response_schema`, and validates the JSON back into a typed object - so every document yields the same shape and you store it without regex over prose. The `legible` boolean is the uncertainty escape hatch.

**How the code works - step by step**
- **`from google.genai import types` / `from pydantic import BaseModel`** - the config type and the schema base class.
- **`class Receipt(BaseModel)`** - declares the target shape: `vendor`, `date`, `total`, and `legible` (False when the model can't read it clearly).
- **`contents=["Extract the receipt fields. Set legible=false if unsure.", Image.open(...)]`** - the instruction plus the image, reusing `client`.
- **`config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Receipt)`** - forces the reply to be JSON matching the schema.
- **`data = Receipt.model_validate_json(resp.text)`** - parses and validates the JSON straight into a typed object.
- **`print(data.vendor, data.total, data.legible)`** - reads fields off the validated object.

**In one line:** schema-constrained JSON out + a `legible` flag so the model marks unreadable fields instead of hallucinating them.

**What you'll see:** The parsed fields print on one line: "Sunrise Grocers 1240.0 True" - vendor, total as a float, and the legibility flag.

## 4 - Multi-image comparison

Comparing a 'before' and 'after' photo is just more parts in the same `contents` list - label them in the prompt and ask what changed. This cell also leans on the strongest lever against hallucination: telling the model to say 'unsure' for anything it cannot clearly see.

> **Needs a Google API key** and `before.jpg` / `after.jpg` files.

In [ ]:
# Multi-image: pass several images in one contents list.
# reuse `client` from the setup
from PIL import Image

before = Image.open("before.jpg")
after  = Image.open("after.jpg")

resp = client.models.generate_content(
    model="gemini-3-pro",      # pro: comparison reasoning is worth the stronger model
    contents=[
        "Image 1 is 'before', image 2 is 'after'. What changed? Be specific but "
        "say 'unsure' for anything you cannot clearly see.",
        before, after,
    ],
)
print(resp.text)
# Output: The 'after' image adds a scratch along the left door panel; the
# Output: wheel and background look unchanged. (Exact scratch length: unsure.)

**Explanation**

This cell scales the same call to two images at once and reaches for the stronger `gemini-3-pro` tier because comparison reasoning is genuinely harder. It labels each image in the prompt so the model knows which is which, and it bakes in an 'unsure' instruction so exact-but-uncertain details (like a scratch's length) get flagged rather than fabricated.

**How the code works - step by step**
- **`before = Image.open(...)` / `after = Image.open(...)`** - loads both images; `client` reused.
- **`model="gemini-3-pro"`** - the stronger tier, chosen because change-detection reasoning is worth it.
- **`contents=[instruction, before, after]`** - the prompt labels image 1 as 'before' and image 2 as 'after', then asks what changed and to say 'unsure' when unclear.
- **`print(resp.text)`** - the comparison answer, with an explicit uncertainty note.

**In one line:** pass several labeled images in one `contents` list, and require an 'unsure' escape hatch for exact details.

**What you'll see:** A specific diff plus a hedge, e.g. it reports a new scratch on the left door panel, says the wheel and background look unchanged, and marks the exact scratch length as "unsure".

## 5 - Self-host an open VLM for private data

Same "image + text in, text out" idea, but the model now runs on your own GPU via `transformers` - so private documents never leave your infrastructure. This is the closed-vs-open tradeoff made concrete: more setup, full data control.

> **Needs a GPU** - loads Qwen3-VL-8B locally and generates on-device; shown for reference.

In [ ]:
# SELF-HOST an open VLM for private data (needs a GPU). Same idea, your infra.
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image

model_id = "Qwen/Qwen3-VL-8B-Instruct"      # open, self-hostable
model = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype="auto",
                                                    device_map="auto")
proc  = AutoProcessor.from_pretrained(model_id)

messages = [{"role": "user", "content": [
    {"type": "image", "image": Image.open("report.jpg")},
    {"type": "text",  "text": "Summarize this medical report in 3 bullet points."},
]}]
inputs = proc.apply_chat_template(messages, add_generation_prompt=True,
                                  tokenize=True, return_dict=True,
                                  return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=256)
# generate() returns prompt + new tokens, so slice the prompt off before decoding
gen = out[0][inputs["input_ids"].shape[-1]:]
print(proc.decode(gen, skip_special_tokens=True))
# Output: - Patient stable; blood pressure within the normal range ...

**Explanation**

This cell swaps the hosted API for a self-hosted open model while keeping the task identical. It downloads Qwen3-VL and its processor, builds a chat-style message that interleaves an image and a text instruction, runs local generation, and - the load-bearing detail - slices the prompt tokens off the output before decoding so you print only the model's new text, not your own question echoed back.

**How the code works - step by step**
- **`from transformers import AutoModelForImageTextToText, AutoProcessor`** - the model and its multimodal processor.
- **`model_id = "Qwen/Qwen3-VL-8B-Instruct"`** - an open, self-hostable VLM; the 8B size fits on a single modern GPU.
- **`AutoModelForImageTextToText.from_pretrained(..., device_map="auto")`** - loads the weights and places them across available GPUs.
- **`messages = [...]`** - a chat message whose `content` interleaves an `image` part and a `text` instruction, mirroring the hosted `contents` list.
- **`proc.apply_chat_template(..., return_dict=True, return_tensors="pt")`** - formats the prompt; `return_dict=True` keeps the image tensors so `**inputs` actually carries the picture.
- **`out = model.generate(**inputs, max_new_tokens=256)`** - runs local generation, returning prompt + new tokens.
- **`gen = out[0][inputs["input_ids"].shape[-1]:]`** - slices off the prompt so only the newly generated tokens remain.
- **`print(proc.decode(gen, skip_special_tokens=True))`** - decodes the answer to text.

**In one line:** same image+text task, but weights on your GPU - build a chat message, generate, slice off the prompt, decode.

**What you'll see:** A short bullet summary of the report from the local model, e.g. beginning "- Patient stable; blood pressure within the normal range ...".

Across six cells the pattern never changed: an image is just another "part" you send next to your text, and the model answers in words. You saw the control surface move up the stack - a sharper prompt (Cell 2), a Pydantic schema with an `legible` escape hatch (Cell 3), labeled multi-image parts (Cell 4) - and then swapped the hosted `gemini-3` client for a self-hosted Qwen3-VL on your own GPU (Cell 5) without changing the shape of the task. The through-line: trust the VLM's holistic judgments, but structure and verify anything exact (counts, positions, totals, tiny text). Next, Lesson 6.6 flips this around - generating across time and space rather than understanding it - and Module 8 gives multimodal RAG its full treatment.